Import required Libraries

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

Load project utilities & Initialize notebook

In [0]:
%run /Workspace/project1/setup_catalogs/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text('catalog', 'project1_cat', 'catalog')
dbutils.widgets.text('data_source', 'products', 'data_source')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

In [0]:
base_path = f's3://sportsbar-childpro/{data_source}/*.csv'
print(base_path)

### Bronze

In [0]:
df = (spark.read.format('csv')
  .option('header', True)
  .option('inferSchema', True)
  .load(base_path)
  .withColumn('read_timestamp', F.current_timestamp()))

In [0]:
df.printSchema()

In [0]:
display(df)

In [0]:
df.write\
.format('delta')\
.option('delta.enableChangeDataFeed', 'true')\
.mode('overwrite')\
.saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

### Silver

In [0]:
df_bronze = spark.read.table(f'{catalog}.{bronze_schema}.{data_source}')
display(df_bronze.limit(10))

### Transformation

Drop Duplicates

In [0]:
df_duplicates = df_bronze.groupBy('product_id').count().filter(F.col('count') > 1)
display(df_duplicates)

In [0]:
df_silver = df_bronze.dropDuplicates(['product_id'])
display(df_silver)

In [0]:
df_duplicates = df_silver.groupBy('product_id').count().filter(F.col('count') > 1)
display(df_duplicates)

Title case fix

energy bar -> Energy Bar

In [0]:
display(df_silver.select('category').distinct())

In [0]:
df_silver = df_silver.withColumn('Category', F.when(F.col('category').isNull(),None).otherwise(F.initcap(F.col('category'))))

In [0]:
display(df_silver)

Fix the spelling mistake 'Protien' to 'Protein'

In [0]:
display(df_silver.withColumn('category', F.regexp_replace(F.col('Category'), '(?i)protien', 'Protein')))

In [0]:
display(df_silver.withColumn('product_name', F.regexp_replace(F.col('product_name'), '(?i)protien', 'Protein')))

In [0]:
df_silver = df_silver.withColumn('category', F.regexp_replace(F.col('category'), '(?i)protien', 'Protein'))
df_silver = df_silver.withColumn('product_name', F.regexp_replace(F.col('product_name'), '(?i)protien', 'Protein'))
display(df_silver)

In [0]:
display(df_silver.select('category').distinct())

In [0]:
# division column
df_silver = df_silver.withColumn('division', 
                                 F.when(F.col('category') == 'Energy Bars', 'Nutrition Bars')
                                 .when(F.col('category') == 'Protein Bars', 'Nutrition Bars')
                                 .when(F.col('category') == 'Granola & Cereals', 'Breakfast Foods')
                                 .when(F.col('category') == 'Recovery Dairy', 'Dairy & Recovery')
                                 .when(F.col('category') == 'Healthy Snacks', 'Healthy Snacks')
                                 .when(F.col('category') == 'Electrolyte Mix', 'Hydration & Electrolytes')
                                 .otherwise('Others'))

In [0]:
# variant column

df_silver = df_silver.withColumn('variant', F.regexp_extract(F.col('product_name'), r'\((.*?)\)', 1))

In [0]:
# product_code column

# Invalid product_id are replaced with a fallback value to avoid losing fact records and ensure downstream joins remain consistent

df_silver = (
    df_silver.withColumn('product_code', F.sha2(F.col('product_name').cast('string'), 256))

# clean the product_id and set only numeric id, else set as null

.withColumn('product_id', F.when(F.col('product_id').cast('string').rlike('^[0-9]+$'), F.col('product_id').cast('string')).otherwise(F.lit(99999).cast('string')))

# Rename preduct_name to product
.withColumnRenamed('product_name', 'product')
)

In [0]:
display(df_silver)

In [0]:
df_silver = df_silver.select('product_code', 'division', 'category', 'product', 'variant', 'product_id', 'read_timestamp')

In [0]:
display(df_silver)

In [0]:
df_silver.write\
.format('delta')\
.option('delta.enableChangeDataFeed', 'true')\
.option('mergeSchema', 'true')\
.mode('overwrite')\
.saveAsTable(f'{catalog}.{silver_schema}.{data_source}')

### Gold

In [0]:
df_silver = spark.sql(f'select * from {catalog}.{silver_schema}.{data_source}')

df_gold = df_silver.select('product_code', 'product_id', 'division', 'category', 'product', 'variant')
display(df_gold.limit(5))

In [0]:
df_gold.write\
.format('delta')\
.option('delta.enableChangeDataFeed', 'true')\
.option('mergeSchema', 'true')\
.mode('overwrite')\
.saveAsTable(f'{catalog}.{gold_schema}.sb_dim_{data_source}')

### Merge data source with parent

In [0]:
delta_table = DeltaTable.forName(spark, 'project1_cat.gold.dim_products')
df_child_products = spark.sql(f'select product_code, division, category, product, variant from project1_cat.gold.sb_dim_products;')
df_child_products.show(5)

In [0]:
# The error happened because you are using pyspark.sql.functions.F.col inside the DeltaTable.merge() API,
# which expects column names as strings, not as Column objects.
# Replace F.col('source.division') with 'source.division', etc.

delta_table.alias('target').merge(
    source=df_child_products.alias('source'),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        'division' : 'source.division',
        'category' : 'source.category',
        'product'  : 'source.product',
        'variant'  : 'source.variant'
    }
).whenNotMatchedInsert(
    values={
        'product_code' : 'source.product_code',
        'division' : 'source.division',
        'category' : 'source.category',
        'product'  : 'source.product',
        'variant'  : 'source.variant'
    }
).execute()